# Exercise 1: Prompt Chaining for Customer Support AI**Goal:** Build a multi-step flow where output of one prompt becomes input to the next.

## Setup: Install Required Libraries

In [ ]:
!pip install anthropic -q

## Implementation: 4-Step Customer Support Chain

In [ ]:
import anthropicimport jsonclient = anthropic.Anthropic()# Define the four promptsSTEP1_PROMPT = '''You are a customer support triage agent. Analyze the customer issue and classify it.Customer Issue:{customer_issue}Classify the issue into ONE of these categories:- Technical Issue- Billing Issue- Account Access Issue- Feature Request- Product Quality IssueRespond with ONLY the category name, nothing else.'''STEP2_PROMPT = '''You are a customer support agent gathering diagnostic information. Based on the issue category and description, ask clarifying questions.Issue Category: {issue_category}Customer Issue: {customer_issue}Generate 2-3 specific clarifying questions relevant to this {issue_category}.Format as a numbered list.Keep questions direct and actionable.'''STEP3_PROMPT = '''You are an expert customer support specialist proposing solutions.Based on the issue category and clarifying questions, suggest the most likely resolution.Issue Category: {issue_category}Customer Issue: {customer_issue}Clarifying Questions Asked:{clarifying_questions}Propose a step-by-step solution (2-4 steps).Be concise and actionable.'''STEP4_PROMPT = '''You are an escalation decision agent. Based on the issue and proposed solution, determine if this should be escalated to a human specialist.Issue Category: {issue_category}Customer Issue: {customer_issue}Proposed Solution:{proposed_solution}Respond with ONLY:- "ESCALATE: [reason]" if human specialist needed- "RESOLVE: [brief summary of next steps]" if resolution is clear'''print("✓ Prompts loaded")

## Define the Chain Execution Function

In [ ]:
def run_prompt_chain(customer_issue):    '''Execute the complete prompt chain'''        print("\n" + "="*70)    print("PROMPT CHAINING FOR CUSTOMER SUPPORT")    print("="*70)    print(f"\nCUSTOMER ISSUE:\n{customer_issue}\n")        # STEP 1: Classify    print("[STEP 1] Classifying Issue...")    response1 = client.messages.create(        model="claude-3-5-sonnet-20241022",        max_tokens=100,        messages=[{"role": "user", "content": STEP1_PROMPT.format(customer_issue=customer_issue)}]    )    issue_category = response1.content[0].text.strip()    print(f"✓ Classification: {issue_category}")        # STEP 2: Gather Info (uses Step 1 output)    print("\n[STEP 2] Gathering Missing Information...")    response2 = client.messages.create(        model="claude-3-5-sonnet-20241022",        max_tokens=300,        messages=[{"role": "user", "content": STEP2_PROMPT.format(            issue_category=issue_category,            customer_issue=customer_issue        )}]    )    clarifying_questions = response2.content[0].text.strip()    print(f"✓ Questions Generated:\n{clarifying_questions}")        # STEP 3: Propose Solution (uses Steps 1 & 2)    print("\n[STEP 3] Proposing Solution...")    response3 = client.messages.create(        model="claude-3-5-sonnet-20241022",        max_tokens=400,        messages=[{"role": "user", "content": STEP3_PROMPT.format(            issue_category=issue_category,            customer_issue=customer_issue,            clarifying_questions=clarifying_questions        )}]    )    proposed_solution = response3.content[0].text.strip()    print(f"✓ Solution Proposed:\n{proposed_solution}")        # STEP 4: Escalation (uses all prior outputs)    print("\n[STEP 4] Determining Escalation...")    response4 = client.messages.create(        model="claude-3-5-sonnet-20241022",        max_tokens=200,        messages=[{"role": "user", "content": STEP4_PROMPT.format(            issue_category=issue_category,            customer_issue=customer_issue,            proposed_solution=proposed_solution        )}]    )    escalation_decision = response4.content[0].text.strip()    print(f"✓ Decision: {escalation_decision}")    print("\n" + "="*70)        return {        "issue": customer_issue,        "classification": issue_category,        "questions": clarifying_questions,        "solution": proposed_solution,        "decision": escalation_decision    }print("✓ Chain function ready")

## Test with Example Customer Issue

In [ ]:
test_issue = '''I've been trying to reset my password for the past 2 hours but I keep getting an error message that says 'Reset link invalid'. I received the email with the reset link, clicked it immediately, but nothing happens. I'm locked out of my account and need access URGENTLY as I have critical work due tomorrow.'''result = run_prompt_chain(test_issue)

======================================================================PROMPT CHAINING FOR CUSTOMER SUPPORT AI======================================================================CUSTOMER ISSUE:    I've been trying to reset my password for the past 2 hours but I keep getting     an error message that says 'Reset link invalid'. I received the email with the     reset link, clicked it immediately, but nothing happens. I'm locked out of my account     and need access URGENTLY as I have critical work due tomorrow.    ----------------------------------------------------------------------[STEP 1] Classifying Issue...✓ Classification: Account Access Issue[STEP 2] Gathering Missing Information...✓ Questions Generated:1. Have you tried clearing your browser cache and cookies before attempting the password reset?2. Are you attempting the password reset from a different device or internet connection?3. Have you checked your email spam/junk folder for the reset link, or tried requesting a new rese

## Results Summary

In [ ]:
print("\n📊 CHAIN EXECUTION SUMMARY")print("="*70)print(f"Classification: {result['classification']}")print(f"\nQuestions Asked:")print(result['questions'])print(f"\nProposed Solution:")print(result['solution'])print(f"\nFinal Decision:")print(result['decision'])